<a href="https://www.kaggle.com/code/ugokorubeast/ugoko-byu?scriptVersionId=315947157" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 目次

- [1. 環境設定と入力探索](#sec-setup)
- [2. tomo 抽出](#sec-select)
- [3. 前処理（raw Image -> .npy）](#sec-preprocess)
- [4. CustomDataset と座標→ラベルマップ変換](#sec-dataset)
- [5. モデル構築](#sec-build-model)
- [6. 学習・評価ループ（2-5）](#sec-train)

この目次から各章へ移動できます。

## 1. 環境設定と入力探索
<a id="sec-setup"></a>

ライブラリ読み込み、`BaseConfig`（2-4）で学習設定を一元化、seed 固定、入力ディレクトリ探索を行います。

In [1]:
from pathlib import Path
from dataclasses import dataclass
import random
import hashlib

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===== 2-4: Base configuration class =====
@dataclass
class BaseConfig:
    seed: int = 42
    num_tomos: int = 4
    depth: int = 16
    input_size: int = 96
    batch_size: int = 1
    num_workers: int = 0
    epochs: int = 3
    lr: float = 1e-3
    label_radius: int = 1


CFG = BaseConfig()

# Backward-compatible aliases for existing cells
SEED = CFG.seed
NUM_TOMOS = CFG.num_tomos
DEPTH = CFG.depth
IMG_SIZE = CFG.input_size
BATCH_SIZE = CFG.batch_size
NUM_WORKERS = CFG.num_workers
EPOCHS = CFG.epochs
LR = CFG.lr
LABEL_RADIUS = CFG.label_radius

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ../input, ./input, train_data などから tomo_* ディレクトリがある場所を自動検出
candidate_roots = [
    Path('../input'),
    Path('../input/competitions/byu-locating-bacterial-flagellar-motors-2025/train'),
    Path('./input'),
    Path('train_data'),
    Path('../train_data'),
]

input_root = None
for root in candidate_roots:
    if root.exists() and any(p.is_dir() and p.name.startswith('tomo_') for p in root.iterdir()):
        input_root = root
        break

if input_root is None:
    raise FileNotFoundError(
        "tomo_* ディレクトリが見つかりません。候補: ../input, ./input, train_data, ../train_data"
    )

print(f"[INFO] input_root = {input_root.resolve()}")
print(f"[INFO] CFG = {CFG}")

[INFO] input_root = /Users/yuhaogao/workspace/BYU-competition/train_data
[INFO] CFG = BaseConfig(seed=42, num_tomos=4, depth=16, input_size=96, batch_size=1, num_workers=0, epochs=3, lr=0.001, label_radius=1)


## 2. tomo 抽出
<a id="sec-select"></a>

利用可能な tomo から seed 固定で学習対象を抽出します。

In [ ]:
all_tomos = sorted([
    p for p in input_root.iterdir()
    if p.is_dir() and p.name.startswith('tomo_')
])

if len(all_tomos) < NUM_TOMOS:
    raise ValueError(f"tomo フォルダ数が不足しています: found={len(all_tomos)}, required={NUM_TOMOS}")

# seed 固定で 4 件ランダム抽出
selected_tomos = random.sample(all_tomos, k=NUM_TOMOS)

print('[INFO] selected_tomos:')
for p in selected_tomos:
    print(' -', p.name)

[INFO] selected_tomos:
 - tomo_2c607f
 - tomo_2bb588
 - tomo_4925ee
 - tomo_f6a38a


## 3. 前処理（raw Image -> .npy）
<a id="sec-preprocess"></a>

生データのスライス画像を読み込み、学習で使いやすい `.npy` ボリュームとして保存します。

In [ ]:
processed_root = Path('./processed_tomos')
processed_root.mkdir(parents=True, exist_ok=True)


def load_and_resize_volume(tomo_dir: Path, img_size: int):
    slice_paths = sorted(tomo_dir.glob('slice_*.jpg'))
    if not slice_paths:
        slice_paths = sorted(tomo_dir.glob('*.jpg'))
    if not slice_paths:
        raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

    volume = []
    for path in slice_paths:
        image = Image.open(path).convert('L').resize((img_size, img_size))
        array = np.asarray(image, dtype=np.float32) / 255.0
        volume.append(array)

    return np.stack(volume, axis=0)  # [D, H, W]


# 2-1: 生データ -> .npy 前処理
# NOTE: まずは選択した tomo のみを保存。全件処理するなら preprocess_targets = all_tomos に変更。
preprocess_targets = selected_tomos

for tomo_dir in preprocess_targets:
    volume = load_and_resize_volume(tomo_dir, IMG_SIZE)
    out_path = processed_root / f'{tomo_dir.name}.npy'
    np.save(out_path, volume.astype(np.float32))
    print(f'[PREPROCESS] {tomo_dir.name} -> {out_path} shape={volume.shape}')

print(f'[INFO] processed_root = {processed_root.resolve()}')

[PREPROCESS] tomo_2c607f -> processed_tomos/tomo_2c607f.npy shape=(300, 96, 96)
[PREPROCESS] tomo_2bb588 -> processed_tomos/tomo_2bb588.npy shape=(300, 96, 96)
[PREPROCESS] tomo_4925ee -> processed_tomos/tomo_4925ee.npy shape=(300, 96, 96)
[PREPROCESS] tomo_f6a38a -> processed_tomos/tomo_f6a38a.npy shape=(500, 96, 96)
[INFO] processed_root = /Users/yuhaogao/workspace/BYU-competition/processed_tomos


## 4. CustomDataset と座標→ラベルマップ変換
<a id="sec-dataset"></a>

CSV 座標を読み取り、3D ラベルマップへ変換して input/target を返す Dataset を作成し、`get_dataset` / `get_dataloader` で DataLoader を組み立てます。

In [ ]:
class CustomDataset(Dataset):
    """2-2 実装: 座標アノテーションを 3D ラベルマップへ変換して返す。"""

    def __init__(self, tomo_dirs, depth=16, img_size=96, label_radius=1, processed_root: Path | None = None):
        self.tomo_dirs = list(tomo_dirs)
        self.depth = depth
        self.img_size = img_size
        self.label_radius = label_radius
        self.processed_root = processed_root
        self.coord_meta = self._load_coord_metadata()

    def __len__(self):
        return len(self.tomo_dirs)

    def _load_slice(self, path: Path):
        img = Image.open(path).convert('L').resize((self.img_size, self.img_size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return arr

    def _load_coord_metadata(self):
        csv_candidates = [
            Path('./folds_all.csv'),
            Path('./data/processed/folds_all.csv'),
            Path('../input/competitions/byu-locating-bacterial-flagellar-motors-2025/train_labels.csv'),
        ]
        csv_path = next((p for p in csv_candidates if p.exists()), None)
        if csv_path is None:
            print('[WARN] folds_all.csv が見つからないため、ラベルは全て 0 になります。')
            return {}

        df = pd.read_csv(csv_path)
        grouped = {}
        for tomo_id, g in df.groupby('tomo_id'):
            first = g.iloc[0]
            coords = []
            # 列名に空白があるため、属性アクセスではなく列参照で座標を取得する
            for z, y, x in g[['Motor axis 0', 'Motor axis 1', 'Motor axis 2']].to_numpy(dtype=float):
                if z >= 0 and y >= 0 and x >= 0:
                    coords.append((z, y, x))

            grouped[tomo_id] = {
                'orig_shape': (
                    int(first['Array shape (axis 0)']),
                    int(first['Array shape (axis 1)']),
                    int(first['Array shape (axis 2)']),
                ),
                'coords': coords,
            }
        return grouped

    def _scale_coord(self, coord, src_len, dst_len):
        if src_len <= 1:
            return 0
        return int(round(coord * (dst_len - 1) / (src_len - 1)))

    def _load_volume_from_images(self, tomo_dir: Path):
        slices = sorted(tomo_dir.glob('slice_*.jpg'))
        if not slices:
            slices = sorted(tomo_dir.glob('*.jpg'))
        if not slices:
            raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

        # z 方向は全体から等間隔サンプリングして depth に揃える
        if len(slices) >= self.depth:
            sample_idx = np.linspace(0, len(slices) - 1, self.depth).astype(int)
            selected = [slices[i] for i in sample_idx]
        else:
            selected = slices + [slices[-1]] * (self.depth - len(slices))

        return np.stack([self._load_slice(p) for p in selected], axis=0)  # [D, H, W]

    def _match_depth(self, vol):
        d = vol.shape[0]
        if d == self.depth:
            return vol
        if d > self.depth:
            idx = np.linspace(0, d - 1, self.depth).astype(int)
            return vol[idx]
        pad = np.repeat(vol[-1:, :, :], self.depth - d, axis=0)
        return np.concatenate([vol, pad], axis=0)

    def _load_volume(self, tomo_dir: Path):
        # 2-1 の出力を優先して読み込み、ない場合のみ raw 画像から作る
        if self.processed_root is not None:
            npy_path = self.processed_root / f'{tomo_dir.name}.npy'
            if npy_path.exists():
                vol = np.load(npy_path).astype(np.float32)
                return self._match_depth(vol)

        return self._load_volume_from_images(tomo_dir)

    def _make_label_map(self, tomo_id: str):
        label = np.zeros((self.depth, self.img_size, self.img_size), dtype=np.float32)
        meta = self.coord_meta.get(tomo_id)
        if meta is None or len(meta['coords']) == 0:
            return label

        src_d, src_h, src_w = meta['orig_shape']
        r = self.label_radius
        for z0, y0, x0 in meta['coords']:
            z = self._scale_coord(z0, src_d, self.depth)
            y = self._scale_coord(y0, src_h, self.img_size)
            x = self._scale_coord(x0, src_w, self.img_size)

            z_min, z_max = max(0, z - r), min(self.depth, z + r + 1)
            y_min, y_max = max(0, y - r), min(self.img_size, y + r + 1)
            x_min, x_max = max(0, x - r), min(self.img_size, x + r + 1)
            label[z_min:z_max, y_min:y_max, x_min:x_max] = 1.0

        return label

    def __getitem__(self, idx):
        tomo_dir = self.tomo_dirs[idx]
        vol = self._load_volume(tomo_dir)
        label = self._make_label_map(tomo_dir.name)

        x = torch.from_numpy(vol).unsqueeze(0)      # [1, D, H, W]
        target = torch.from_numpy(label).unsqueeze(0)  # [1, D, H, W]
        return {'input': x, 'target': target, 'tomo_id': tomo_dir.name}


# 2-3: get_dataset / get_dataloader 実装
# mode ごとの既定値を分けつつ、shuffle/batch/num_workers を明示的に扱う
DATALOADER_CFG = {
    'train': {
        'batch_size': BATCH_SIZE,
        'shuffle': True,
        'num_workers': NUM_WORKERS,
    },
    'val': {
        'batch_size': BATCH_SIZE,
        'shuffle': False,
        'num_workers': NUM_WORKERS,
    },
}


def get_dataset(tomo_dirs, mode='train'):
    return CustomDataset(
        tomo_dirs,
        depth=DEPTH,
        img_size=IMG_SIZE,
        label_radius=LABEL_RADIUS,
        processed_root=processed_root,
    )


def get_dataloader(dataset, mode='train', shuffle=None, batch_size=None, num_workers=None):
    cfg = DATALOADER_CFG.get(mode, DATALOADER_CFG['train'])

    if shuffle is None:
        shuffle = cfg['shuffle']
    if batch_size is None:
        batch_size = cfg['batch_size']
    if num_workers is None:
        num_workers = cfg['num_workers']

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )


# 2-5: train/valid 比較のための最小 split（固定順）
if len(selected_tomos) >= 2:
    train_tomos = selected_tomos[:-1]
    val_tomos = selected_tomos[-1:]
else:
    train_tomos = selected_tomos
    val_tomos = selected_tomos

train_ds = get_dataset(train_tomos, mode='train')
val_ds = get_dataset(val_tomos, mode='val')

train_loader = get_dataloader(train_ds, mode='train')
val_loader = get_dataloader(val_ds, mode='val')

sample = next(iter(train_loader))
print('[INFO] train_loader cfg =', DATALOADER_CFG['train'])
print('[INFO] val_loader cfg =', DATALOADER_CFG['val'])
print('[INFO] train_tomos =', [p.name for p in train_tomos])
print('[INFO] val_tomos =', [p.name for p in val_tomos])
print('[INFO] sample input shape =', tuple(sample['input'].shape))
print('[INFO] sample target shape =', tuple(sample['target'].shape))
print('[INFO] sample target positive voxels =', int(sample['target'].sum().item()))

[INFO] train_loader cfg = {'batch_size': 1, 'shuffle': True, 'num_workers': 0}
[INFO] val_loader cfg = {'batch_size': 1, 'shuffle': False, 'num_workers': 0}
[INFO] train_tomos = ['tomo_2c607f', 'tomo_2bb588', 'tomo_4925ee']
[INFO] val_tomos = ['tomo_f6a38a']
[INFO] sample input shape = (1, 1, 16, 96, 96)
[INFO] sample target shape = (1, 1, 16, 96, 96)
[INFO] sample target positive voxels = 27


## 5. モデル構築
<a id="sec-build-model"></a>

In [2]:
# ===== 3-1-3: BasicBlock downsample branch implementation =====
from timm.layers import DropPath


def conv3x3x3(ic, oc, stride=1):
    return nn.Conv3d(
        in_channels=ic,
        out_channels=oc,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
    )


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=False, drop_path_rate=0.0):
        super().__init__()
        self.conv1 = conv3x3x3(inplanes, planes, stride=stride)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3x3(planes, planes, stride=1)
        self.bn2 = nn.BatchNorm3d(planes)
        self.drop_path = DropPath(drop_prob=drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()

        if isinstance(downsample, nn.Module):
            self.downsample = downsample
        elif downsample:
            self.downsample = nn.Sequential(
                nn.Conv3d(
                    inplanes,
                    planes * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm3d(planes * self.expansion),
            )
        else:
            self.downsample = None

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.drop_path(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = out + residual
        out = self.relu(out)
        return out


# 3-1-3 validation: downsample=True with stride/channel change
x_ds = torch.randn(2, 16, 8, 32, 32)
block_ds = BasicBlock(inplanes=16, planes=32, stride=2, downsample=True, drop_path_rate=0.1)
block_ds.eval()
y_ds = block_ds(x_ds)
print(f"[downsample] input shape:  {tuple(x_ds.shape)}")
print(f"[downsample] output shape: {tuple(y_ds.shape)}")
assert y_ds.shape == (2, 32, 4, 16, 16)

/Users/yuhaogao/.pyenv/versions/3.10.11/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[downsample] input shape:  (2, 16, 8, 32, 32)
[downsample] output shape: (2, 32, 4, 16, 16)


In [3]:
# ===== 3-1-4: Bottleneck block implementation =====
from timm.layers import DropPath


class Bottleneck(nn.Module):
    """1x1x1 -> 3x3x3 -> 1x1x1 bottleneck residual block."""

    def __init__(self, inplanes, planes, stride=1, downsample=False, expansion_factor=4, drop_path_rate=0.0):
        super().__init__()
        out_channels = planes * expansion_factor

        # 1x1x1 reduction
        self.conv1 = nn.Conv3d(inplanes, planes, kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)

        # 3x3x3 spatial conv (stride applied here)
        self.conv2 = nn.Conv3d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)

        # 1x1x1 expansion
        self.conv3 = nn.Conv3d(planes, out_channels, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm3d(out_channels)

        self.relu = nn.ReLU(inplace=True)
        self.drop_path = DropPath(drop_prob=drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()

        if isinstance(downsample, nn.Module):
            self.downsample = downsample
        elif downsample:
            self.downsample = nn.Sequential(
                nn.Conv3d(inplanes, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels),
            )
        else:
            self.downsample = None

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)
        out = self.drop_path(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = out + residual
        out = self.relu(out)
        return out


# 3-1-4 validation A: no downsample (inplanes == planes * expansion_factor, stride=1)
x_bn = torch.randn(2, 64, 8, 32, 32)
bottleneck_no_ds = Bottleneck(
    inplanes=64,
    planes=16,  # out = 16 * expansion_factor = 64
    stride=1,
    downsample=False,
    expansion_factor=4,
    drop_path_rate=0.1,
).eval()
y_bn = bottleneck_no_ds(x_bn)
print(f"[bottleneck no_ds] input:  {tuple(x_bn.shape)}")
print(f"[bottleneck no_ds] output: {tuple(y_bn.shape)}")
assert y_bn.shape == x_bn.shape, "downsample=False では shape が一致する必要があります"

# 3-1-4 validation B: downsample with stride/channel change
x_bn_ds = torch.randn(2, 64, 8, 32, 32)
bottleneck_ds = Bottleneck(
    inplanes=64,
    planes=32,  # out = 32 * expansion_factor = 128
    stride=2,
    downsample=True,
    expansion_factor=4,
    drop_path_rate=0.1,
).eval()
y_bn_ds = bottleneck_ds(x_bn_ds)
print(f"[bottleneck ds] input:  {tuple(x_bn_ds.shape)}")
print(f"[bottleneck ds] output: {tuple(y_bn_ds.shape)}")
assert y_bn_ds.shape == (2, 128, 4, 16, 16), "expansion後チャネルとresidual shape を合わせる必要があります"

[bottleneck no_ds] input:  (2, 64, 8, 32, 32)
[bottleneck no_ds] output: (2, 64, 8, 32, 32)
[bottleneck ds] input:  (2, 64, 8, 32, 32)
[bottleneck ds] output: (2, 128, 4, 16, 16)


In [4]:
# ===== 3-1-6/3-1-7/3-1-8: backbone table + stem + _make_layer =====
from types import SimpleNamespace


def conv_out_dim(in_size, kernel_size, stride=1, padding=0, dilation=1):
    return ((in_size + 2 * padding - dilation * (kernel_size - 1) - 1) // stride) + 1


class ResnetEncoder3d(nn.Module):
    """Encoder scaffold with backbone config table, stem, and stage builder."""

    BACKBONE_TABLE = {
        "r3d18": {"layers": [2, 2, 2, 2], "block": BasicBlock},
        "r3d200": {"layers": [3, 24, 36, 3], "block": Bottleneck},
    }

    def __init__(self, cfg, in_stride=(2, 2, 2), in_dilation=(1, 1, 1)):
        super().__init__()
        self.cfg = cfg
        self.in_stride = in_stride
        self.in_dilation = in_dilation

        backbone_name = cfg.backbone
        if backbone_name not in self.BACKBONE_TABLE:
            supported = ", ".join(self.BACKBONE_TABLE.keys())
            raise ValueError(f"ResnetEncoder3d backbone: {backbone_name} not implemented. supported=[{supported}]")

        spec = self.BACKBONE_TABLE[backbone_name]
        self.layers = spec["layers"]
        self.block = spec["block"]

        in_padding = tuple(d * 3 for d in in_dilation)
        self.conv1 = nn.Conv3d(
            in_channels=cfg.in_chans,
            out_channels=64,
            kernel_size=(7, 7, 7),
            stride=in_stride,
            dilation=in_dilation,
            padding=in_padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=2, padding=1)

        # 3-1-8: build layer1..layer4 as nn.Sequential
        self.inplanes = 64
        self.layer1 = self._make_layer(self.block, planes=64, n_blocks=self.layers[0], stride=1)
        self.layer2 = self._make_layer(self.block, planes=128, n_blocks=self.layers[1], stride=2)
        self.layer3 = self._make_layer(self.block, planes=256, n_blocks=self.layers[2], stride=2)
        self.layer4 = self._make_layer(self.block, planes=512, n_blocks=self.layers[3], stride=2)

    def _block_expansion(self):
        return 4 if self.block is Bottleneck else 1

    def _make_block(self, block, inplanes, planes, stride, downsample):
        if block is Bottleneck:
            return block(
                inplanes=inplanes,
                planes=planes,
                stride=stride,
                downsample=downsample,
                expansion_factor=4,
            )
        return block(
            inplanes=inplanes,
            planes=planes,
            stride=stride,
            downsample=downsample,
        )

    def _make_layer(self, block, planes, n_blocks, stride=1):
        expansion = self._block_expansion()
        out_channels = planes * expansion

        # first block handles stride/downsample when shape changes
        need_downsample = (stride != 1) or (self.inplanes != out_channels)
        downsample = None
        if need_downsample:
            downsample = nn.Sequential(
                nn.Conv3d(self.inplanes, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels),
            )

        layers = [
            self._make_block(
                block=block,
                inplanes=self.inplanes,
                planes=planes,
                stride=stride,
                downsample=downsample,
            )
        ]

        # subsequent blocks keep normal residual connection (stride=1, no projection)
        self.inplanes = out_channels
        for _ in range(1, n_blocks):
            layers.append(
                self._make_block(
                    block=block,
                    inplanes=self.inplanes,
                    planes=planes,
                    stride=1,
                    downsample=None,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x


def resolve_backbone(cfg):
    model = ResnetEncoder3d(cfg)
    return model.layers, model.block.__name__


# validation A: r3d18 backbone table
cfg18 = SimpleNamespace(backbone="r3d18", in_chans=1)
layers18, block18 = resolve_backbone(cfg18)
print(f"[r3d18] layers={layers18}, block={block18}")
assert layers18 == [2, 2, 2, 2]
assert block18 == "BasicBlock"

# validation B: r3d200 backbone table
cfg200 = SimpleNamespace(backbone="r3d200", in_chans=1)
layers200, block200 = resolve_backbone(cfg200)
print(f"[r3d200] layers={layers200}, block={block200}")
assert layers200 == [3, 24, 36, 3]
assert block200 == "Bottleneck"

# validation C: unsupported backbone should raise ValueError
try:
    _ = ResnetEncoder3d(SimpleNamespace(backbone="r3d999", in_chans=1))
    raise AssertionError("unsupported backbone で ValueError が必要です")
except ValueError as e:
    print(f"[unsupported] expected error: {e}")

# validation D: layer1..4 are nn.Sequential and block counts match config
encoder18 = ResnetEncoder3d(cfg18).eval()
assert isinstance(encoder18.layer1, nn.Sequential)
assert isinstance(encoder18.layer2, nn.Sequential)
assert isinstance(encoder18.layer3, nn.Sequential)
assert isinstance(encoder18.layer4, nn.Sequential)
assert len(encoder18.layer1) == 2
assert len(encoder18.layer2) == 2
assert len(encoder18.layer3) == 2
assert len(encoder18.layer4) == 2
print("[r3d18] layer blocks:", [len(encoder18.layer1), len(encoder18.layer2), len(encoder18.layer3), len(encoder18.layer4)])

encoder200 = ResnetEncoder3d(cfg200).eval()
assert isinstance(encoder200.layer1, nn.Sequential)
assert isinstance(encoder200.layer2, nn.Sequential)
assert isinstance(encoder200.layer3, nn.Sequential)
assert isinstance(encoder200.layer4, nn.Sequential)
assert len(encoder200.layer1) == 3
assert len(encoder200.layer2) == 24
assert len(encoder200.layer3) == 36
assert len(encoder200.layer4) == 3
print("[r3d200] layer blocks:", [len(encoder200.layer1), len(encoder200.layer2), len(encoder200.layer3), len(encoder200.layer4)])

# validation E: stem shape shrink check
x_stem = torch.randn(2, 1, 32, 96, 96)
y_stem = encoder18.maxpool(encoder18.relu(encoder18.bn1(encoder18.conv1(x_stem))))
conv_d = conv_out_dim(32, kernel_size=7, stride=2, padding=3, dilation=1)
conv_h = conv_out_dim(96, kernel_size=7, stride=2, padding=3, dilation=1)
conv_w = conv_out_dim(96, kernel_size=7, stride=2, padding=3, dilation=1)
pool_d = conv_out_dim(conv_d, kernel_size=3, stride=2, padding=1, dilation=1)
pool_h = conv_out_dim(conv_h, kernel_size=3, stride=2, padding=1, dilation=1)
pool_w = conv_out_dim(conv_w, kernel_size=3, stride=2, padding=1, dilation=1)
expected_shape = (2, 64, pool_d, pool_h, pool_w)
print(f"[encoder stem] input shape:  {tuple(x_stem.shape)}")
print(f"[encoder stem] output shape: {tuple(y_stem.shape)}")
print(f"[encoder stem] expected:     {expected_shape}")
assert y_stem.shape == expected_shape, "encoder 内 stem の shape が期待値と一致する必要があります"

[r3d18] layers=[2, 2, 2, 2], block=BasicBlock
[r3d200] layers=[3, 24, 36, 3], block=Bottleneck
[unsupported] expected error: ResnetEncoder3d backbone: r3d999 not implemented. supported=[r3d18, r3d200]
[r3d18] layer blocks: [2, 2, 2, 2]
[r3d200] layer blocks: [3, 24, 36, 3]
[encoder stem] input shape:  (2, 1, 32, 96, 96)
[encoder stem] output shape: (2, 64, 8, 24, 24)
[encoder stem] expected:     (2, 64, 8, 24, 24)


## 6. 学習・評価ループ（2-5）
<a id="sec-train"></a>

train/valid を分けて loss と最小評価指標（precision, recall, fbeta）を比較します。

In [ ]:
class Tiny3DAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y


def binary_metrics_from_logits_or_probs(pred, target, threshold=0.5, beta=2.0):
    pred_bin = (pred >= threshold).to(torch.int64)
    tgt_bin = (target >= 0.5).to(torch.int64)

    tp = ((pred_bin == 1) & (tgt_bin == 1)).sum().item()
    fp = ((pred_bin == 1) & (tgt_bin == 0)).sum().item()
    fn = ((pred_bin == 0) & (tgt_bin == 1)).sum().item()

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    beta2 = beta * beta
    fbeta = (1 + beta2) * precision * recall / max(1e-8, beta2 * precision + recall)

    return {
        'precision': precision,
        'recall': recall,
        'fbeta': fbeta,
    }


def run_one_epoch(model, loader, optimizer, criterion, device, train=True):
    if train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    metric_sum = {'precision': 0.0, 'recall': 0.0, 'fbeta': 0.0}
    n_steps = 0

    for step, batch in enumerate(loader, start=1):
        x = batch['input'].to(device)
        target = batch['target'].to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            y = model(x)
            loss = criterion(y, target)
            if train:
                loss.backward()
                optimizer.step()

        metrics = binary_metrics_from_logits_or_probs(y.detach(), target.detach(), threshold=0.5, beta=2.0)
        running_loss += loss.item()
        metric_sum['precision'] += metrics['precision']
        metric_sum['recall'] += metrics['recall']
        metric_sum['fbeta'] += metrics['fbeta']
        n_steps += 1

        mode = 'train' if train else 'valid'
        print(
            f'[{mode}] step={step}/{len(loader)} loss={loss.item():.6f} '
            f'P={metrics["precision"]:.4f} R={metrics["recall"]:.4f} F2={metrics["fbeta"]:.4f}'
        )

    epoch_loss = running_loss / max(1, n_steps)
    epoch_metrics = {k: v / max(1, n_steps) for k, v in metric_sum.items()}
    return epoch_loss, epoch_metrics


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Tiny3DAutoEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

print(f'[INFO] device = {device}')
print(f'[INFO] train_steps_per_epoch = {len(train_loader)}')
print(f'[INFO] valid_steps_per_epoch = {len(val_loader)}')

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_m = run_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        train=True,
    )
    val_loss, val_m = run_one_epoch(
        model=model,
        loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        train=False,
    )

    current_lr = optimizer.param_groups[0]['lr']
    row = {
        'epoch': epoch,
        'lr': current_lr,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_precision': train_m['precision'],
        'val_precision': val_m['precision'],
        'train_recall': train_m['recall'],
        'val_recall': val_m['recall'],
        'train_fbeta': train_m['fbeta'],
        'val_fbeta': val_m['fbeta'],
    }
    history.append(row)

    print(
        f'[epoch] {epoch} lr={current_lr:.2e} '
        f'train_loss={train_loss:.6f} val_loss={val_loss:.6f} '
        f'train_F2={train_m["fbeta"]:.4f} val_F2={val_m["fbeta"]:.4f}'
    )

history_df = pd.DataFrame(history)
display(history_df)

[INFO] device = cpu
[INFO] train_steps_per_epoch = 3
[INFO] valid_steps_per_epoch = 1
[train] step=1/3 loss=0.716268 P=0.0000 R=0.0000 F2=0.0000
[train] step=2/3 loss=0.712193 P=0.0002 R=1.0000 F2=0.0009
[train] step=3/3 loss=0.707416 P=0.0002 R=1.0000 F2=0.0010
[valid] step=1/1 loss=0.701552 P=0.0007 R=0.8942 F2=0.0035
[epoch] 1 lr=1.00e-03 train_loss=0.711959 val_loss=0.701552 train_F2=0.0006 val_F2=0.0035
[train] step=1/3 loss=0.701815 P=0.0002 R=1.0000 F2=0.0010
[train] step=2/3 loss=0.695607 P=0.0002 R=0.7778 F2=0.0011
[train] step=3/3 loss=0.684308 P=0.0000 R=0.0000 F2=0.0000
[valid] step=1/1 loss=0.671082 P=0.0000 R=0.0000 F2=0.0000
[epoch] 2 lr=1.00e-03 train_loss=0.693910 val_loss=0.671082 train_F2=0.0007 val_F2=0.0000
[train] step=1/3 loss=0.671730 P=0.0001 R=0.0370 F2=0.0004
[train] step=2/3 loss=0.658511 P=0.0000 R=0.0000 F2=0.0000
[train] step=3/3 loss=0.630833 P=0.0000 R=0.0000 F2=0.0000
[valid] step=1/1 loss=0.600669 P=0.0000 R=0.0000 F2=0.0000
[epoch] 3 lr=1.00e-03 trai

,epoch,lr,train_loss,val_loss,train_precision,val_precision,train_recall,val_recall,train_fbeta,val_fbeta
0,1,0.001,0.711959,0.701552,0.000125,0.000706,0.666667,0.894231,0.000623,0.003518
1,2,0.001,0.693910,0.671082,0.000140,0.000000,0.592593,0.000000,0.000701,0.000000
2,3,0.001,0.653691,0.600669,0.000030,0.000000,0.012346,0.000000,0.000150,0.000000
